In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/23 12:03:23 WARN Utils: Your hostname, Adrians-M2-Macbook.local, resolves to a loopback address: 127.0.0.1; using 172.16.102.39 instead (on interface en0)
26/03/23 12:03:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 12:03:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
!ls ../data/homework/raw/

yellow_tripdata_2025-11.parquet


In [16]:
df_raw = spark.read\
    .parquet("../data/homework/raw/")

In [20]:
df_raw.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [26]:
df_raw.coalesce(4) \
    .write \
    .mode("overwrite") \
    .parquet("../data/homework/parquet/")

In [27]:
df = spark.read.parquet("../data/homework/parquet/")

In [28]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [ ]:
df\
    .withColumn("pickup_date", F.to_date(F.col("tpep_pickup_datetime")))\
    .filter(F.col("pickup_date") == '2025-11-15') \
    .groupBy(F.col("pickup_date")) \
    .count()\
    .show()


+-----------+------+
|pickup_date| count|
+-----------+------+
| 2025-11-15|162604|
+-----------+------+



In [ ]:
duration_hours = F.round(
    ( F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime") )  / 3600,
    2
)

df \
    .select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance") \
    .withColumn("duration_hours", duration_hours) \
    .orderBy("duration_hours", ascending=False) \
    .show(5)

+--------+--------------------+---------------------+-------------+--------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_hours|
+--------+--------------------+---------------------+-------------+--------------+
|       2| 2025-11-26 20:22:12|  2025-11-30 15:01:00|       121.17|         90.65|
|       2| 2025-11-27 04:22:41|  2025-11-30 09:19:35|         1.08|         76.95|
|       2| 2025-11-03 10:42:55|  2025-11-06 14:55:45|          0.0|         76.21|
|       2| 2025-11-07 11:23:22|  2025-11-10 08:40:41|         6.54|         69.29|
|       2| 2025-11-18 17:12:47|  2025-11-21 12:17:37|         0.76|         67.08|
+--------+--------------------+---------------------+-------------+--------------+
only showing top 5 rows


In [66]:
df_zones = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/homework/raw/zones/")



In [67]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [88]:
df \
    .select("PULocationID") \
    .groupBy(F.col("PULocationID")) \
    .agg(F.count("*").alias("ride_count")) \
    .join(df_zones, on= F.col("PULocationID").cast("int") == F.col("LocationID").cast("int"), how="left") \
    .select("PULocationID", "Zone", "ride_count") \
    .orderBy("ride_count", ascending=True) \
    .show(5, truncate=False)


+------------+---------------------------------------------+----------+
|PULocationID|Zone                                         |ride_count|
+------------+---------------------------------------------+----------+
|105         |Governor's Island/Ellis Island/Liberty Island|1         |
|5           |Arden Heights                                |1         |
|84          |Eltingville/Annadale/Prince's Bay            |1         |
|187         |Port Richmond                                |3         |
|109         |Great Kills                                  |4         |
+------------+---------------------------------------------+----------+
only showing top 5 rows


In [83]:
df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [84]:
df_zones.dtypes

[('LocationID', 'int'),
 ('Borough', 'string'),
 ('Zone', 'string'),
 ('service_zone', 'string')]